<div align="center">

# Quantum Computing Hackathon &mdash; Track 1

### Qiskit Fall Fest / Quantum Hackathon
**RIFT Labs (Research in Future Technologies)**
**The Catholic University of America, College of Engineering, Physics, and Computing**

</div>

---

In [1]:
# =====================================================================
#  SETUP  (run this first, it is the only cell that needs the internet)
# =====================================================================
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    """Install pkg only if it is not already importable."""
    name = import_name or pkg
    try:
        importlib.import_module(name)
    except ImportError:
        print(f"installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    return importlib.import_module(name)

ensure("qiskit")
ensure("numpy")

import numpy as np
import qiskit
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import (Statevector, DensityMatrix, Operator,
                                 SparsePauliOp, Pauli, partial_trace,
                                 state_fidelity)

print("qiskit", qiskit.__version__, "| numpy", np.__version__)

# ----- tiny toolkit used throughout (no plotting libraries required) ---
rng = np.random.default_rng(7)
_SCORE = []


def check(label, ok, detail=""):
    """Record and print a pass/fail line. Never raises, so a wrong answer
    does not stop the notebook."""
    _SCORE.append((label, bool(ok)))
    print(("[ PASS ] " if ok else "[ FAIL ] ") + label + (("   " + str(detail)) if detail else ""))
    return bool(ok)


def scoreboard():
    """Summarise every check() run so far."""
    if not _SCORE:
        print("no checks run yet")
        return
    passed = sum(1 for _, ok in _SCORE if ok)
    print("=" * 58)
    print(f"  {passed} / {len(_SCORE)} checks passed")
    print("=" * 58)
    for label, ok in _SCORE:
        print(("  ok   " if ok else "  FAIL ") + label)
    if passed != len(_SCORE):
        raise AssertionError(f"{len(_SCORE) - passed} check(s) failed")


def bar(counts, width=40, title=None, top=None):
    """ASCII histogram of a counts dict. Works in any environment."""
    if title:
        print(title)
    if not counts:
        print("  (no data)")
        return
    items = sorted(counts.items(), key=lambda kv: -kv[1])
    if top:
        items = items[:top]
    total = sum(counts.values())
    hi = max(counts.values())
    for k, v in sorted(items):
        n = int(round(width * v / hi))
        print(f"  |{k}>  {'#' * n:<{width}} {v:>6}  {100 * v / total:5.1f}%")


def sample(circ, shots=2048):
    """Shot-based measurement without needing a backend at all: build the exact
    statevector, then draw multinomial samples from it. Any final measure()
    instructions are stripped first, so you can pass either form."""
    clean = circ.remove_final_measurements(inplace=False)
    sv = Statevector.from_instruction(clean)
    return {str(k): int(v) for k, v in sv.sample_counts(shots).items()}


def bloch(circ):
    """Return (<X>, <Y>, <Z>) for a single-qubit circuit."""
    sv = Statevector.from_instruction(circ)
    return tuple(float(np.real(sv.expectation_value(Pauli(p)))) for p in "XYZ")


def pretty(sv, tol=1e-9):
    """Readable ket expansion of a Statevector."""
    n = sv.num_qubits
    terms = []
    for i, a in enumerate(sv.data):
        if abs(a) > tol:
            re, im = np.real(a), np.imag(a)
            c = f"{re:+.4f}" if abs(im) < tol else f"({re:+.4f}{im:+.4f}j)"
            terms.append(f"{c}|{format(i, '0' + str(n) + 'b')}>")
    return "  ".join(terms) if terms else "0"

qiskit 2.5.2 | numpy 2.4.4


### A sanity check on the toolkit

The cell below should print a Bell state and a histogram with only `00` and `11` in it.
If that works, your environment is good and you can start.

In [ ]:
demo = QuantumCircuit(2)
demo.h(0)
demo.cx(0, 1)
print("state:", pretty(Statevector(demo)))
bar(sample(demo, shots=1000), title="\n1000 shots of a Bell state:")

---
# Problem 1. Your first qubit, and loading a probability into it

**Difficulty: warm-up**

### Background

A single qubit lives in a two dimensional complex vector space. Its state is

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \qquad |\alpha|^2 + |\beta|^2 = 1.$$

When you measure in the computational basis you get outcome 0 with probability
$|\alpha|^2$ and outcome 1 with probability $|\beta|^2$. The state itself is never
observed, only one of those two outcomes, so to see the probabilities you have to
repeat the experiment many times. Those repeats are called *shots*.

The Hadamard gate takes $|0\rangle$ to an even superposition:

$$H = \frac{1}{\sqrt 2}\begin{pmatrix} 1 & 1 \\ 1 & -1\end{pmatrix},
\qquad H|0\rangle = \frac{|0\rangle + |1\rangle}{\sqrt 2}.$$

Measure that and you get 0 half the time and 1 half the time. But a Hadamard only ever
gives you 50/50. To get an *arbitrary* bias you need a rotation. The $R_y$ gate rotates
about the $y$ axis of the Bloch sphere:

$$R_y(\theta) = \begin{pmatrix} \cos\tfrac{\theta}{2} & -\sin\tfrac{\theta}{2} \\
\sin\tfrac{\theta}{2} & \cos\tfrac{\theta}{2}\end{pmatrix},
\qquad R_y(\theta)|0\rangle = \cos\tfrac{\theta}{2}|0\rangle + \sin\tfrac{\theta}{2}|1\rangle.$$

So the probability of measuring 1 is $\sin^2(\theta/2)$. Note that $H = R_y(\pi/2)$ up
to a phase, which is exactly the 50/50 case.

### Your task

Write `biased_qubit(p)` that returns a one qubit circuit whose measurement gives
outcome 1 with probability exactly `p`, for any `p` in [0, 1].

Invert the relation above: you want $\sin^2(\theta/2) = p$, so solve for $\theta$.

In [3]:
def biased_qubit(p):
    """Return a 1-qubit QuantumCircuit with P(measure 1) == p."""
    qc = QuantumCircuit(1)
    # YOUR CODE HERE: pick theta so that sin^2(theta/2) == p, then qc.ry(theta, 0)
    return qc

In [ ]:
# ---- check -----------------------------------------------------------
ok = True
for p in [0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 1.0]:
    probs = Statevector(biased_qubit(p)).probabilities()
    ok &= abs(probs[1] - p) < 1e-10
check("P1  biased_qubit reproduces every requested probability", ok)

bar(sample(biased_qubit(0.25), shots=4000), title="\n4000 shots at p = 0.25:")
print("\nNote the shot noise: you asked for 25.0% and you got something near it.")
print("Statistical error falls off as 1/sqrt(shots), so 4x the shots halves the error.")

---
# Problem 2. The Bloch sphere, and reading a qubit three ways

**Difficulty: warm-up**

### Background

A pure single qubit state is a point on the surface of a sphere. Write it as

$$|\psi\rangle = \cos\tfrac{\theta}{2}|0\rangle + e^{i\phi}\sin\tfrac{\theta}{2}|1\rangle$$

and the point is at polar angle $\theta$ and azimuth $\phi$. The Cartesian coordinates
of that point are the three **Pauli expectation values**:

$$r_x = \langle\psi|X|\psi\rangle, \quad r_y = \langle\psi|Y|\psi\rangle,
\quad r_z = \langle\psi|Z|\psi\rangle$$

with

$$X = \begin{pmatrix}0&1\\1&0\end{pmatrix},\quad
Y = \begin{pmatrix}0&-i\\i&0\end{pmatrix},\quad
Z = \begin{pmatrix}1&0\\0&-1\end{pmatrix}.$$

For a pure state $r_x^2 + r_y^2 + r_z^2 = 1$, which is what puts it on the surface.
Mixed states sit strictly inside.

Why three numbers? Because a single measurement axis is not enough to pin a qubit down.
$|0\rangle$ and $|1\rangle$ differ in $r_z$. But $|+\rangle = (|0\rangle+|1\rangle)/\sqrt2$
and $|-\rangle = (|0\rangle-|1\rangle)/\sqrt2$ have **identical** $Z$ statistics (50/50
either way) and are told apart only by $r_x = \pm 1$. The phase is invisible in the
computational basis. That single fact is behind most of the algorithms later in this
notebook.

Two gates you need: $R_y(\theta)$ from Problem 1 tips the state away from the north
pole, and $R_z(\phi)$ spins it about the vertical axis,

$$R_z(\phi) = \begin{pmatrix} e^{-i\phi/2} & 0 \\ 0 & e^{+i\phi/2}\end{pmatrix}.$$

### Your task

Write `prepare_bloch(theta, phi)` returning a one qubit circuit whose Bloch vector is

$$(\sin\theta\cos\phi,\; \sin\theta\sin\phi,\; \cos\theta).$$

Starting from $|0\rangle$ (which sits at the north pole, $r_z = +1$), apply the two
rotations in the right order. Order matters: rotations do not commute.

In [ ]:
def prepare_bloch(theta, phi):
    """Return a 1-qubit circuit whose Bloch vector is
    (sin(theta)cos(phi), sin(theta)sin(phi), cos(theta))."""
    qc = QuantumCircuit(1)
    # YOUR CODE HERE: one ry and one rz, in the correct order
    return qc

In [ ]:
# ---- check -----------------------------------------------------------
ok = True
for theta, phi in [(0, 0), (np.pi / 2, 0), (np.pi / 2, np.pi / 2),
                   (np.pi, 0), (0.7, 2.1), (2.5, -1.3)]:
    want = (np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta))
    got = bloch(prepare_bloch(theta, phi))
    ok &= np.allclose(got, want, atol=1e-10)
check("P2  prepare_bloch hits every requested Bloch vector", ok)

print("\nThe phase is invisible to a Z measurement:")
for name, qc in [("|+>", prepare_bloch(np.pi / 2, 0)),
                 ("|->", prepare_bloch(np.pi / 2, np.pi)),
                 ("|i>", prepare_bloch(np.pi / 2, np.pi / 2))]:
    rx, ry, rz = bloch(qc)
    p = Statevector(qc).probabilities()
    print(f"  {name}:  <X>={rx:+.3f}  <Y>={ry:+.3f}  <Z>={rz:+.3f}"
          f"   P(0)={p[0]:.3f} P(1)={p[1]:.3f}")
print("\nAll three give 50/50 in the computational basis. Only <X> and <Y> tell them apart.")

---
# Problem 3. Entanglement and the four Bell states

**Difficulty: easy**

### Background

Two qubits live in a four dimensional space spanned by $|00\rangle, |01\rangle,
|10\rangle, |11\rangle$. Most two qubit states **cannot** be written as a product
$|a\rangle \otimes |b\rangle$ of two separate single qubit states. Those are the
entangled ones, and the four maximally entangled two qubit states are the Bell states:

$$|\Phi^{+}\rangle = \frac{|00\rangle + |11\rangle}{\sqrt 2}, \qquad
  |\Phi^{-}\rangle = \frac{|00\rangle - |11\rangle}{\sqrt 2},$$
$$|\Psi^{+}\rangle = \frac{|01\rangle + |10\rangle}{\sqrt 2}, \qquad
  |\Psi^{-}\rangle = \frac{|01\rangle - |10\rangle}{\sqrt 2}.$$

The recipe for $|\Phi^{+}\rangle$ is two gates: a Hadamard on qubit 0, then a CNOT with
qubit 0 as control and qubit 1 as target. CNOT flips the target when the control is 1:

$$\mathrm{CNOT}|00\rangle = |00\rangle, \quad \mathrm{CNOT}|10\rangle = |11\rangle.$$

Feed it a superposition of control values and you get a superposition of *correlated*
pairs, which is the whole trick.

To reach the other three, nudge the inputs or outputs. An $X$ on one qubit swaps
$|\Phi\rangle \leftrightarrow |\Psi\rangle$ because it flips one bit of each term.
A $Z$ on one qubit flips the sign of the term where that qubit is 1, turning
$+ \to -$.

### Why they are a good test of entanglement

For $|\Phi^{+}\rangle$, both $\langle Z_0 Z_1\rangle = +1$ and
$\langle X_0 X_1\rangle = +1$. Perfect correlation in *two different bases at once*
is impossible for any classical pair of coins, and impossible for any product state.
Each qubit on its own, though, is completely random: its reduced state is
$\tfrac12 I$. The information lives in the correlations, not in the parts.

### Your task

Write `bell(k)` returning the two qubit circuit for Bell state number `k`, in the order
$k = 0, 1, 2, 3 \to |\Phi^{+}\rangle, |\Phi^{-}\rangle, |\Psi^{+}\rangle, |\Psi^{-}\rangle$.

Remember Qiskit's bit ordering: in `|01>` qubit 0 is the 1.

In [ ]:
BELL_TARGETS = {
    0: np.array([1, 0, 0,  1]) / np.sqrt(2),   # |Phi+> = (|00> + |11>)/sqrt2
    1: np.array([1, 0, 0, -1]) / np.sqrt(2),   # |Phi-> = (|00> - |11>)/sqrt2
    2: np.array([0, 1, 1,  0]) / np.sqrt(2),   # |Psi+> = (|01> + |10>)/sqrt2
    3: np.array([0, 1, -1, 0]) / np.sqrt(2),   # |Psi-> = (|01> - |10>)/sqrt2
}

def bell(k):
    """Return the 2-qubit circuit preparing Bell state k (0..3)."""
    qc = QuantumCircuit(2)
    # YOUR CODE HERE: h + cx for k=0, then extra x / z to reach k=1,2,3
    return qc

In [ ]:
# ---- check -----------------------------------------------------------
names = ["|Phi+>", "|Phi->", "|Psi+>", "|Psi->"]
ok = True
for k in range(4):
    got = Statevector(bell(k)).data
    # compare up to an irrelevant global phase
    overlap = abs(np.vdot(BELL_TARGETS[k], got))
    ok &= overlap > 1 - 1e-10
check("P3  all four Bell states are prepared correctly", ok)

print()
for k in range(4):
    sv = Statevector(bell(k))
    zz = np.real(sv.expectation_value(SparsePauliOp("ZZ")))
    xx = np.real(sv.expectation_value(SparsePauliOp("XX")))
    print(f"  {names[k]:7s} {pretty(sv):44s}  <ZZ>={zz:+.1f}  <XX>={xx:+.1f}")

rho0 = partial_trace(Statevector(bell(0)), [1])
print(f"\nReduced state of qubit 0 in |Phi+>:\n{np.round(rho0.data, 3)}")
check("P3  each half of a Bell pair on its own is maximally mixed",
      np.allclose(rho0.data, np.eye(2) / 2, atol=1e-12))

---
# Problem 4. Scaling up: GHZ states and parity

**Difficulty: easy to moderate**

### Background

The $n$ qubit generalisation of $|\Phi^{+}\rangle$ is the GHZ state,

$$|\mathrm{GHZ}_n\rangle = \frac{|00\cdots0\rangle + |11\cdots1\rangle}{\sqrt 2}.$$

Either every qubit reads 0 or every qubit reads 1, with nothing in between. The obvious
circuit is a Hadamard on qubit 0 followed by a chain of CNOTs, qubit 0 to 1, then 1 to 2,
and so on. That chain has **depth $n$**, because each CNOT waits for the one before it.

On real hardware depth is what kills you: every extra layer of gates is more time for
the state to decohere. There is a better construction. After the Hadamard, qubit 0 holds
the superposition, and you can copy it to qubit 1; now **two** qubits hold it and can
each copy to a fresh qubit in the same layer; then four, then eight. That doubling gives
depth $O(\log n)$ instead of $O(n)$, and it is a pattern worth having in your head
because it shows up all over quantum circuit design.

### Measuring parity

For GHZ the useful observables are products of Paulis. Two facts to verify:

- $\langle Z_0 Z_j \rangle = +1$ for every $j$, because the two branches are
  all-zeros and all-ones, so any pair of qubits always agrees.
- $\langle X_0 X_1 \cdots X_{n-1} \rangle = +1$, the global $X$ parity. Each $X$
  flips every bit of a term, so $X^{\otimes n}$ maps $|00\cdots0\rangle$ to
  $|11\cdots1\rangle$ and back, and the symmetric superposition is its $+1$ eigenstate.

Both at once is the $n$ qubit version of the Bell state argument.

### Your task

Write `ghz_linear(n)` with the CNOT chain, and `ghz_log(n)` with the doubling
construction. They must produce the same state, and `ghz_log` must have strictly
smaller depth once $n \ge 4$.

In [ ]:
def ghz_linear(n):
    """n-qubit GHZ via a CNOT chain. Depth grows like n."""
    qc = QuantumCircuit(n)
    # YOUR CODE HERE
    return qc

def ghz_log(n):
    """n-qubit GHZ via doubling. Depth grows like log(n)."""
    qc = QuantumCircuit(n)
    # YOUR CODE HERE: h(0), then repeatedly cx(j, j + step) for a halving step
    return qc

In [ ]:
# ---- check -----------------------------------------------------------
ok_state, ok_depth, ok_corr = True, True, True
print("   n   depth(linear)  depth(log)   same state?")
for n in range(2, 13):
    a, b = Statevector(ghz_linear(n)), Statevector(ghz_log(n))
    same = state_fidelity(a, b) > 1 - 1e-10
    da, db = ghz_linear(n).depth(), ghz_log(n).depth()
    ok_state &= same
    if n >= 4:
        ok_depth &= db < da
    print(f"  {n:2d}   {da:7d}      {db:7d}        {same}")

    target = np.zeros(2 ** n, dtype=complex)
    target[0] = target[-1] = 1 / np.sqrt(2)
    ok_state &= abs(np.vdot(target, a.data)) > 1 - 1e-10

    for j in range(1, n):                      # <Z_0 Z_j> == +1
        lab = ["I"] * n
        lab[n - 1 - 0], lab[n - 1 - j] = "Z", "Z"
        ok_corr &= abs(np.real(a.expectation_value(SparsePauliOp("".join(lab)))) - 1) < 1e-10
    ok_corr &= abs(np.real(a.expectation_value(SparsePauliOp("X" * n))) - 1) < 1e-10

check("P4  both constructions give the exact GHZ state", ok_state)
check("P4  doubling beats the chain on depth for n >= 4", ok_depth)
check("P4  <Z0 Zj> = +1 for all j, and <X...X> = +1", ok_corr)

bar(sample(ghz_log(5), shots=2000), title="\n2000 shots of GHZ(5), only two outcomes appear:")

---
# Problem 5. Quantum teleportation

**Difficulty: moderate**

### Background

Alice has a qubit in an unknown state $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$.
She wants Bob to end up holding it. She cannot measure it (that destroys it and gives her
only one bit), and she cannot copy it (the no-cloning theorem forbids a machine that
duplicates an arbitrary unknown state). What she *can* do is spend one pre-shared
entangled pair plus two classical bits.

Label the qubits: `q0` is Alice's unknown payload, `q1` is Alice's half of the Bell pair,
`q2` is Bob's half.

1. Prepare $|\Phi^{+}\rangle$ across `q1` and `q2`.
2. Alice applies `cx(q0, q1)` then `h(q0)`. This is a Bell basis measurement in disguise:
   it rotates the Bell basis into the computational basis.
3. Alice measures `q0` and `q1`, getting two classical bits.
4. Bob applies $X$ if the `q1` bit is 1, then $Z$ if the `q0` bit is 1.

After the correction Bob's qubit is exactly $|\psi\rangle$. Note what did *not* happen:
nothing travelled faster than light, because Bob is stuck until the two classical bits
arrive, and Alice's copy is gone, because her measurement destroyed it.

### Why no measurement appears in the circuit below

Measuring in the middle and then conditioning gates on the result needs a backend that
supports mid-circuit measurement. There is a standard identity that avoids it, the
**principle of deferred measurement**: a measurement followed by a classically
controlled gate gives the same final statistics as a quantum controlled gate followed
by the measurement. So replace "measure `q1`, apply $X$ if 1" with a plain `cx(q1, q2)`,
and "measure `q0`, apply $Z$ if 1" with `cz(q0, q2)`.

That version runs on a pure state simulator. To confirm Bob really holds $|\psi\rangle$
we trace out Alice's two qubits and compare the resulting density matrix against the
original state. If the protocol works, the fidelity is 1 for *every* input, which is the
point: teleportation is state-independent.

### Your task

Write `teleport(payload)`, where `payload` is a one qubit circuit preparing
$|\psi\rangle$. Return a three qubit circuit implementing the deferred-measurement
version, with Bob's qubit as `q2`.

In [ ]:
def teleport(payload):
    """3-qubit teleportation circuit (deferred measurement form).
    q0 = Alice's payload, q1 = Alice's Bell half, q2 = Bob's Bell half."""
    qc = QuantumCircuit(3)
    qc.compose(payload, qubits=[0], inplace=True)   # load |psi> onto q0
    # YOUR CODE HERE:
    #   1. Bell pair across q1, q2
    #   2. Alice's cx(0, 1) then h(0)
    #   3. deferred corrections: cx(1, 2) then cz(0, 2)
    return qc

In [ ]:
# ---- check -----------------------------------------------------------
ok = True
print("  theta    phi      fidelity of Bob's qubit")
trials = [(0, 0), (np.pi, 0), (np.pi / 2, 0), (np.pi / 2, np.pi / 2),
          (0.9, 2.4), (2.2, -0.7), (1.3, 3.0)]
for theta, phi in trials:
    payload = prepare_bloch(theta, phi)
    want = Statevector(payload)
    bob = partial_trace(Statevector(teleport(payload)), [0, 1])
    f = state_fidelity(want, bob)
    ok &= f > 1 - 1e-10
    print(f"  {theta:5.2f}  {phi:+5.2f}      {f:.12f}")
check("P5  teleportation is exact for every input state", ok)

print("\nAnd a random state, to be sure it is not the test cases:")
rnd = QuantumCircuit(1)
rnd.ry(float(rng.uniform(0, np.pi)), 0)
rnd.rz(float(rng.uniform(0, 2 * np.pi)), 0)
f = state_fidelity(Statevector(rnd), partial_trace(Statevector(teleport(rnd)), [0, 1]))
check("P5  teleportation is state independent", f > 1 - 1e-10, f"fidelity = {f:.12f}")

---
# Problem 6. Deutsch and Jozsa: one query instead of many

**Difficulty: moderate**

### Background

Someone hands you a black box computing $f: \{0,1\}^n \to \{0,1\}$, and promises it is
either **constant** (same output for all $2^n$ inputs) or **balanced** (0 on exactly
half the inputs, 1 on the other half). Which is it?

Classically, in the worst case, you need $2^{n-1} + 1$ queries. You can get unlucky and
see the same answer over and over from a balanced function until you have checked just
over half the inputs. A quantum computer settles it in **one** query, with certainty.

### The phase oracle

Give the box its input in superposition and have it write its answer into a *phase*
rather than into a register:

$$U_f : |x\rangle \mapsto (-1)^{f(x)}|x\rangle.$$

Then the algorithm is three steps on $n$ qubits:

1. $H^{\otimes n}$ on $|0\rangle^{\otimes n}$, giving
   $\tfrac{1}{\sqrt{2^n}}\sum_x |x\rangle$, an equal superposition over every input.
2. Apply $U_f$, giving $\tfrac{1}{\sqrt{2^n}}\sum_x (-1)^{f(x)}|x\rangle$.
3. $H^{\otimes n}$ again, then measure.

The amplitude of the all-zeros outcome after step 3 works out to
$\tfrac{1}{2^n}\sum_x (-1)^{f(x)}$. If $f$ is constant that sum is $\pm 2^n$, so the
amplitude is $\pm 1$ and you measure all zeros with **probability 1**. If $f$ is
balanced the $+1$s and $-1$s cancel exactly, the amplitude is 0, and you measure all
zeros with **probability 0**. One query, one bit of output, no ambiguity.

The second round of Hadamards is the part worth staring at. It turns the phase pattern
stamped across the superposition into an interference pattern you can actually read. The
same move drives Problems 7 and 8.

### Your task

Write `phase_oracle(bits)` turning a truth table into a phase oracle (apply $Z$-type
phases so basis state $|x\rangle$ picks up $(-1)^{f(x)}$), then `deutsch_jozsa(bits)`
returning the full circuit.

A clean way to build the oracle for an arbitrary truth table: use a diagonal unitary
whose entries are $(-1)^{f(x)}$. Qiskit's `DiagonalGate` does exactly that.
For the two special cases you can also do it with plain gates, which is a good exercise:
$f(x) = x_0$ is a single `z(0)`, and $f(x) = x_0 \oplus x_1 \oplus \cdots$ is a `z` on
every qubit.

In [ ]:
def constant_f(n, value):
    """Truth table of a constant function on n bits."""
    return [value] * (2 ** n)

def balanced_f(n, mask=None):
    """Truth table of f(x) = parity of (x AND mask), which is balanced
    for any nonzero mask."""
    mask = mask if mask is not None else 1
    return [bin(x & mask).count("1") % 2 for x in range(2 ** n)]

def phase_oracle(bits):
    """Circuit applying |x> -> (-1)^bits[x] |x>."""
    n = int(np.log2(len(bits)))
    qc = QuantumCircuit(n)
    # YOUR CODE HERE
    return qc

def deutsch_jozsa(bits):
    """Full Deutsch-Jozsa circuit for the given truth table."""
    n = int(np.log2(len(bits)))
    qc = QuantumCircuit(n)
    # YOUR CODE HERE: H layer, oracle, H layer
    return qc

In [ ]:
# ---- check -----------------------------------------------------------
ok = True
print("   n   kind        P(all zeros)   verdict")
for n in [2, 3, 4, 5]:
    for kind, bits in [("constant 0", constant_f(n, 0)),
                       ("constant 1", constant_f(n, 1)),
                       ("balanced  ", balanced_f(n, 1)),
                       ("balanced  ", balanced_f(n, (2 ** n) - 1))]:
        p0 = Statevector(deutsch_jozsa(bits)).probabilities()[0]
        verdict = "constant" if p0 > 0.5 else "balanced"
        truth = "constant" if len(set(bits)) == 1 else "balanced"
        ok &= (verdict == truth) and (abs(p0 - round(p0)) < 1e-10)
        print(f"  {n:2d}   {kind}   {p0:12.10f}   {verdict}")

check("P6  one query separates constant from balanced, with certainty", ok)
print("\nEvery probability above is exactly 1 or exactly 0. No repeats needed.")
print(f"Classically, n=5 would take up to {2 ** 4 + 1} queries in the worst case.")

---
# Problem 7. Grover search

**Difficulty: moderate to hard**

### Background

Now the box is a yes/no test over $N = 2^n$ items with exactly one item marked. Classical
search needs $O(N)$ queries on average. Grover's algorithm needs $O(\sqrt N)$, which is
a quadratic speedup and the best any quantum algorithm can do for an unstructured
search (that optimality is itself a theorem).

Think of it as a rotation in a two dimensional plane. Write $|w\rangle$ for the marked
item and $|s\rangle$ for the uniform superposition over all items. One Grover iteration
is two reflections:

1. **The oracle** $U_w = I - 2|w\rangle\langle w|$ flips the sign of the marked item.
2. **The diffuser** $U_s = 2|s\rangle\langle s| - I$ reflects about the uniform state.
   In gates: Hadamards, flip the sign of $|0\cdots0\rangle$, Hadamards again.

Two reflections make a rotation. Each iteration turns the state by $2\theta$ toward
$|w\rangle$, where $\sin\theta = 1/\sqrt N$. Starting at angle $\theta$ from $|s\rangle$,
after $k$ iterations the success probability is

$$P(k) = \sin^2\!\big((2k+1)\theta\big).$$

### The part people get wrong

More iterations is not better. $P(k)$ is a sine squared, so it climbs to nearly 1 and
then **comes back down**. The right number of iterations is

$$k^{\star} = \left\lfloor \frac{\pi}{4}\sqrt{N} \right\rfloor,$$

and overshooting walks you back out of the answer. Your check below will show this
explicitly, and it is the single most useful intuition to carry out of this problem.

### Building the pieces

For the oracle, a controlled-$Z$ across all $n$ qubits flips the sign of
$|11\cdots1\rangle$ only. To mark a different bitstring $w$, sandwich it in $X$ gates:
put an $X$ on every qubit where $w$ has a 0, so that $w$ maps to all ones, apply the
multi-controlled $Z$, then undo the $X$s. Qiskit spells the multi-controlled $Z$ as
`qc.mcp(np.pi, control_qubits, target_qubit)`.

The diffuser is the same trick with $w = 0\cdots0$, wrapped in Hadamards.

### Your task

Write `grover_oracle(n, w)`, `diffuser(n)`, and `grover(n, w, iterations=None)`, where
`iterations=None` means use $k^{\star}$.

In [ ]:
def mcz(qc, qubits):
    """Multi-controlled Z on the given qubits: flips the sign of |11...1>."""
    if len(qubits) == 1:
        qc.z(qubits[0])
    else:
        qc.mcp(np.pi, list(qubits[:-1]), qubits[-1])

def grover_oracle(n, w):
    """Flip the sign of basis state w only. w is a string like '101'
    (leftmost character is qubit n-1, matching Qiskit's bit order)."""
    qc = QuantumCircuit(n)
    # YOUR CODE HERE: X where w has a 0, mcz, undo the X gates
    return qc

def diffuser(n):
    """Reflection about the uniform superposition."""
    qc = QuantumCircuit(n)
    # YOUR CODE HERE
    return qc

def grover(n, w, iterations=None):
    """Full Grover circuit. iterations=None uses floor(pi/4 * sqrt(N))."""
    N = 2 ** n
    k = int(np.floor(np.pi / 4 * np.sqrt(N))) if iterations is None else iterations
    qc = QuantumCircuit(n)
    # YOUR CODE HERE: H layer, then k times (oracle + diffuser)
    return qc

In [ ]:
# ---- check 1: does it find the right item, for every target? ---------
ok = True
for n in [2, 3, 4, 5]:
    N = 2 ** n
    worst = 1.0
    for x in range(N):
        w = format(x, "0" + str(n) + "b")
        probs = Statevector(grover(n, w)).probabilities()
        worst = min(worst, probs[x])
        ok &= int(np.argmax(probs)) == x
    print(f"  n={n}  N={N:3d}  k*={int(np.floor(np.pi / 4 * np.sqrt(N))):2d}"
          f"   worst-case success over all targets = {worst:.4f}")
check("P7  Grover peaks on the marked item for every target and every n in 2..5", ok)

In [ ]:
# ---- check 2: the overshoot, which is the real lesson ----------------
n, w = 4, "1011"
target = int(w, 2)
N = 2 ** n
theta = np.arcsin(1 / np.sqrt(N))
kstar = int(np.floor(np.pi / 4 * np.sqrt(N)))

print(f"n={n}, N={N}, marked item |{w}> , k* = {kstar}\n")
print("   k   P(marked) measured   sin^2((2k+1)theta)   ")
rows = []
for k in range(0, 13):
    p = Statevector(grover(n, w, iterations=k)).probabilities()[target]
    pred = np.sin((2 * k + 1) * theta) ** 2
    rows.append(p)
    flag = "  <-- k*" if k == kstar else ("  <-- overshot" if k > kstar and p < rows[kstar] else "")
    print(f"  {k:2d}       {p:.6f}            {pred:.6f}{flag}")

check("P7  the analytic formula sin^2((2k+1)theta) matches the simulation",
      all(abs(rows[k] - np.sin((2 * k + 1) * theta) ** 2) < 1e-9 for k in range(13)))
check("P7  k* is the best choice among the first few iterations",
      int(np.argmax(rows[:6])) == kstar, f"argmax(k<6) = {int(np.argmax(rows[:6]))}, k* = {kstar}")
check("P7  overshooting the peak lowers the success probability",
      rows[kstar + 3] < rows[kstar],
      f"P({kstar}) = {rows[kstar]:.4f} but P({kstar + 3}) = {rows[kstar + 3]:.4f}")

revival = int(np.argmax(rows))
print(f"\nOne subtlety: Grover is a rotation, so it is periodic. Over k = 0..12 the")
print(f"global best is actually k = {revival} with P = {rows[revival]:.4f}, a hair above")
print(f"P({kstar}) = {rows[kstar]:.4f}, because the state has swung all the way round")
print(f"again. That is not a free lunch: it costs {revival}/{kstar:.0f} times the oracle")
print("calls to buy a fraction of a percent. k* is the right engineering answer.")

bar(sample(grover(n, w), shots=4000), title="\n4000 shots at k = k*:", top=6)
print("\nAt k = 2k* the state has rotated past the answer and back toward the start.")
bar(sample(grover(n, w, iterations=2 * kstar + 1), shots=4000),
    title=f"\n4000 shots at k = {2 * kstar + 1}, well past the peak:", top=6)

---
# Problem 8. The quantum Fourier transform, and reading a phase out of a gate

**Difficulty: hard, and the capstone**

### Background, part 1: the QFT

The quantum Fourier transform is the quantum analogue of the discrete Fourier transform.
On $n$ qubits it acts on basis states as

$$\mathrm{QFT}|x\rangle = \frac{1}{\sqrt{N}}\sum_{y=0}^{N-1} e^{2\pi i\, x y / N}|y\rangle,
\qquad N = 2^n.$$

The classical FFT costs $O(N \log N)$ operations. The QFT costs $O(n^2) = O(\log^2 N)$
gates, which is exponentially fewer. That does not give you the Fourier coefficients
(they are amplitudes, and you cannot read amplitudes directly), but it is the engine
inside phase estimation, order finding, and Shor's algorithm.

The circuit is a tidy recursion. For qubit $j$ counting down from the most significant:

1. A Hadamard on qubit $j$.
2. A controlled phase $CP(2\pi / 2^{\,k})$ onto qubit $j$ from each less significant
   qubit, with $k$ growing as the controls get further away.
3. At the very end, reverse the qubit order with swaps, because the recursion leaves the
   output bit-reversed.

For $n = 1$ the QFT is just a Hadamard, which is a good consistency check on your
implementation.

### Background, part 2: phase estimation

Here is what the QFT is for. Suppose you have a unitary $U$ and an eigenvector
$|u\rangle$ with

$$U|u\rangle = e^{2\pi i \varphi}|u\rangle, \qquad \varphi \in [0, 1).$$

You want $\varphi$. Take $m$ counting qubits plus the register holding $|u\rangle$:

1. Hadamard the counting register.
2. For each counting qubit $j$, apply $U^{2^{\,j}}$ to the eigenvector register,
   controlled on counting qubit $j$. Because $|u\rangle$ is an eigenvector, the only
   effect is to stamp a phase $e^{2\pi i\, 2^{\,j}\varphi}$ on the $|1\rangle$ branch of
   that counting qubit. The counting register now holds
   $\tfrac{1}{\sqrt{2^m}}\sum_y e^{2\pi i\, y \varphi}|y\rangle$, which is precisely a
   Fourier basis state.
3. Apply the **inverse** QFT to the counting register and measure. You read off the
   binary digits of $\varphi$.

If $\varphi$ happens to be exactly an $m$-bit binary fraction, the answer is exact and
deterministic. Otherwise you get the nearest $m$-bit value with high probability, and
more counting qubits buy more digits.

We will test on $U = P(2\pi\varphi)$, the single qubit phase gate, whose eigenvector is
simply $|1\rangle$. Pick $\varphi = 5/8 = 0.101_2$ and three counting qubits, and the
measurement should come out as exactly `101` every single time.

### Your task

Write `qft(n)` from scratch (no `qiskit.circuit.library.QFT`), then
`phase_estimation(m, phi)` using its inverse.

In [ ]:
def qft(n, do_swaps=True):
    """Quantum Fourier transform on n qubits, built from scratch."""
    qc = QuantumCircuit(n, name="QFT")
    # YOUR CODE HERE:
    #   for j from n-1 down to 0:  h(j), then cp(2*pi/2**(j-k+1), k, j) for k < j
    #   then reverse the qubit order with swaps
    return qc

def phase_estimation(m, phi):
    """m counting qubits (0..m-1) plus one eigenvector qubit (index m).
    U = P(2*pi*phi) acting on the eigenvector |1>."""
    qc = QuantumCircuit(m + 1)
    # YOUR CODE HERE:
    #   x(m) to load the eigenvector, h on the counting qubits,
    #   controlled P(2*pi*phi*2**j) from counting qubit j onto qubit m,
    #   then the inverse QFT on the counting register
    return qc

In [ ]:
# ---- check 1: is it really the Fourier transform? --------------------
def dft_matrix(n):
    N = 2 ** n
    y, x = np.meshgrid(np.arange(N), np.arange(N), indexing="ij")
    return np.exp(2j * np.pi * x * y / N) / np.sqrt(N)

ok = True
for n in range(1, 6):
    got = Operator(qft(n)).data
    ok &= np.allclose(got, dft_matrix(n), atol=1e-10)
check("P8  qft(n) equals the DFT matrix exactly for n = 1..5", ok)

check("P8  qft(1) is a Hadamard",
      np.allclose(Operator(qft(1)).data, np.array([[1, 1], [1, -1]]) / np.sqrt(2)))
check("P8  qft(n) is unitary and its inverse undoes it",
      np.allclose(Operator(qft(4).compose(qft(4).inverse())).data, np.eye(16), atol=1e-10))

# cross-check against the library implementation, as a second opinion
from qiskit.circuit.library import QFT as LibQFT
check("P8  matches qiskit.circuit.library.QFT",
      np.allclose(Operator(qft(4)).data, Operator(LibQFT(4)).data, atol=1e-10))

In [ ]:
# ---- check 2: phase estimation ---------------------------------------
print("Exactly representable phases, 3 counting qubits:\n")
ok = True
for num in range(8):
    phi = num / 8
    sv = Statevector(phase_estimation(3, phi))
    counting = sv.probabilities(list(range(3)))      # marginal over the counting register
    best = int(np.argmax(counting))
    ok &= (best == num) and (counting[best] > 1 - 1e-9)
    print(f"  phi = {num}/8 = {phi:.3f}   ->  measured {format(best, '03b')}"
          f" = {best}/8   with probability {counting[best]:.10f}")
check("P8  phase estimation is exact for every 3-bit phase", ok)

print("\nA phase that is NOT exactly representable, phi = 1/3:\n")
for m in [3, 5, 8]:
    counting = Statevector(phase_estimation(m, 1 / 3)).probabilities(list(range(m)))
    best = int(np.argmax(counting))
    print(f"  m={m:2d} counting qubits  ->  best estimate {best}/{2 ** m}"
          f" = {best / 2 ** m:.6f}   P = {counting[best]:.4f}   (true value 0.333333)")
check("P8  more counting qubits give a better estimate of 1/3",
      abs(int(np.argmax(Statevector(phase_estimation(8, 1 / 3)).probabilities(list(range(8))))) / 256 - 1 / 3)
      < abs(int(np.argmax(Statevector(phase_estimation(3, 1 / 3)).probabilities(list(range(3))))) / 8 - 1 / 3))

print("\nAnd here is what happens if you forget the swaps in the QFT:")
def pe_noswap(m, phi):
    qc = QuantumCircuit(m + 1)
    qc.x(m); qc.h(range(m))
    for j in range(m):
        qc.cp(2 * np.pi * phi * (2 ** j), j, m)
    qc.compose(qft(m, do_swaps=False).inverse(), qubits=list(range(m)), inplace=True)
    return qc
bad = int(np.argmax(Statevector(pe_noswap(3, 5 / 8)).probabilities([0, 1, 2])))
print(f"  phi = 5/8 = 101 in binary, but without swaps you read {format(bad, '03b')}"
      f" = {bad}/8, the bit-reversal.")

---
# Scoreboard

In [ ]:
scoreboard()

---
<div align="center">
<b>RIFT Labs</b> (Research in Future Technologies)<br>
<b>The Catholic University of America</b>, College of Engineering, Physics, and Computing
</div>